# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 colorectal cancer survivors dataset via the Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

---

In [ ]:
# Ensure `mlcroissant` is installed (uncomment the below line if running in a new environment)
!pip install mlcroissant

## 1. Data Loading
Load the Croissant dataset metadata and schema with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata summary (using class attributes, not dict subscripting)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
List available record sets and fields with their `@id`s.

*This helps discover what parts of the data can be loaded and explored. Use these IDs in subsequent steps to refer to record sets and fields.*

In [ ]:
# Get all record sets defined in the metadata
# All Croissant entities, including record sets, must be referenced by `@id`.

record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in dataset metadata. Attempting to enumerate from the full metadata (dataset spec).\n")
    # Some Croissant datasets list record sets as main entities; list top-level and print their fields
    for node in dataset._spec.get('@graph', []):
        if node.get('@type') == 'cr:RecordSet' or node.get('@type') == 'RecordSet':
            print(f"Record set: {node['@id']}")
            if 'cr:field' in node:
                fields = node['cr:field'] if isinstance(node['cr:field'], list) else [node['cr:field']]
                print("  Fields:")
                for f in fields:
                    if isinstance(f, dict) and '@id' in f:
                        print(f"    - {f['@id']}")
                    else:
                        print(f"    - {f}")
            elif 'field' in node:
                # fallback for alternate field key
                fields = node['field'] if isinstance(node['field'], list) else [node['field']]
                print("  Fields:")
                for f in fields:
                    if isinstance(f, dict) and '@id' in f:
                        print(f"    - {f['@id']}")
                    else:
                        print(f"    - {f}")
else:
    for rs in record_sets:
        print(f"Record set: {rs['@id']}  |  Name: {rs.get('name', '')}")
        field_ids = []
        if 'field' in rs:
            field_ids = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        elif 'cr:field' in rs:
            field_ids = rs['cr:field'] if isinstance(rs['cr:field'], list) else [rs['cr:field']]
        print("  Fields:")
        for f_id in field_ids:
            if isinstance(f_id, dict) and '@id' in f_id:
                print(f"    - {f_id['@id']}")
            else:
                print(f"    - {f_id}")

## 3. Data Extraction
Based on the record sets and their `@id`s listed above, select and load actual records for analysis.

Below, we prepare to load all available RecordSets (typically one for tabular data), using their `@id`.

In [ ]:
# Find all record sets (again, from dataset._spec because record_sets may be empty)
croissant_graph = dataset._spec.get('@graph', [])

# List all RecordSet @id's
record_set_ids = [node['@id'] for node in croissant_graph if node.get('@type') in ['cr:RecordSet','RecordSet']]
print(f"Found {len(record_set_ids)} record sets:")
for rs_id in record_set_ids:
    print(f"  - {rs_id}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from RecordSet: {record_set_id}")
    try:
        # Records is a generator of dicts for each record
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"DataFrame shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Choose the main RecordSet for tabular analysis (the one with the largest or most informative set of columns)
if len(dataframes) > 0:
    main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[1])
    print(f"\nUsing main record set: {main_record_set_id}")
    print(f"Sample columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply processing steps to your dataset, such as filtering for records above a threshold, normalizing numeric fields, and grouping and summarizing data by a categorical field.
All columns and groupings are referenced by their `@id`.

In [ ]:
# For demonstration, select a numeric field and a group field, using column names (which are the `@id`s).
df = dataframes[main_record_set_id]
print("Available columns (use their @id for further analysis):")
print(df.columns.tolist())

# Attempt to detect a numeric field (e.g., fields covering age, interval, or tumor size)
import numpy as np
# Pick first numeric column found
numeric_field_id = None
for col in df.columns:
    # Try converting a random sample to float; if succeeds, use as numeric
    col_samples = df[col].dropna().astype(str).head(10)
    try:
        floats = col_samples.astype(float)
        numeric_field_id = col
        break
    except:
        pass
if numeric_field_id is None:
    raise Exception("No numeric field found in record set.")
print(f"Numeric field selected: {numeric_field_id}")

# Let's pick a threshold for filtering (median + 10% for demo)
try:
    values = pd.to_numeric(df[numeric_field_id], errors='coerce')
    thresh = values.median() + 0.1 * values.std()
    print(f"Using threshold {thresh:.3f}")
except:
    thresh = 10

# Filter records above threshold
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > thresh].copy()
print(f"Filtered records with {numeric_field_id} > {thresh:.2f}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
vals = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
filtered_df[f"{numeric_field_id}_normalized"] = (vals - vals.mean()) / vals.std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field (e.g., first non-numeric field with < 10 unique vals)
group_field_id = None
for col in df.columns:
    if col == numeric_field_id: continue
    nunique = df[col].nunique()
    if nunique > 1 and nunique < 10 and not np.issubdtype(df[col].dtype, np.number):
        group_field_id = col
        break
if group_field_id:
    print(f"Grouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
    print(grouped_df.head())

## 5. Visualization
Create simple plots using the selected numeric and group fields to visualize the distribution and grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=15, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

# Barplot for grouped means if group_field_id is defined
if group_field_id:
    plt.figure(figsize=(8,4))
    order = grouped_df.sort_values(f'mean_{numeric_field_id}').index
    sns.barplot(x=grouped_df.index, y=grouped_df[f'mean_{numeric_field_id}'], order=order)
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion

- This notebook demonstrated how to load and analyze the FAIR^2 colorectal cancer survivors dataset using `mlcroissant`.
- All dataset entities (record sets, fields, columns) were referenced by their `@id` as per Croissant conventions.
- We explored data structure, performed records extraction, simple filtering, normalization, grouping, and basic visualization.

You may extend this notebook with deeper domain-specific analyses based on your research questions!